# RTP20 / I20 CNN training

get functions from `scripts/cnn_train.py` for training.

In [ ]:
import sys
import os
import importlib

# add repo root so `scripts` package can be imported
sys.path.append(os.path.abspath("."))
from scripts import cnn_train
importlib.reload(cnn_train)

# Examples:
# 1) Run full training (will use CSVs under krx_daily_dataset/data_kospi/data_kosdaq)
# cnn_train.main()

# 2) Quick dry-run: small epoch and batch for smoke-test
# dfs = cnn_train.load_csvs(["krx_daily_dataset"])  # ensure this folder exists
# records = cnn_train.prepare_records(dfs)
# model = cnn_train.train(records, epochs=1, batch_size=8)

print("Module loaded. Call cnn_train.main() to start training.")

In [ ]:
import sys
import os
import importlib
from collections import Counter

# add repo root so `scripts` package can be imported
sys.path.append(os.path.abspath("."))
from scripts import cnn_train
importlib.reload(cnn_train)

print('Loaded scripts.cnn_train')

# check available data dirs
data_dirs = [d for d in ["krx_daily_dataset", "data_kospi", "data_kosdaq"] if os.path.isdir(d)]
print('Data dirs found:', data_dirs)


In [ ]:
# Data loading and RTP20 check

# Load CSVs from discovered data dirs
if len(data_dirs) == 0:
    print('No data directories found. Create one and add CSVs before running these cells.')
else:
    dfs = cnn_train.load_csvs(data_dirs)
    print(f"Loaded {len(dfs)} tickers")
    # show one sample header
    sample_name = next(iter(dfs.keys()))
    display(dfs[sample_name].head())
    # compute RTP20 for sample and show tail
    dfs[sample_name] = cnn_train.compute_rtp20(dfs[sample_name])
    display(dfs[sample_name][['Date','Close','Volume','RTP20']].tail(10))


In [ ]:
# Image creation demo and dataset preparation

if len(data_dirs) > 0:
    # create small sample of records (may be large depending on number of tickers)
    records = cnn_train.prepare_records(dfs, past_window=20, future_horizon=20)
    print('Total records prepared:', len(records))
    # show example image for first record
    if len(records) > 0:
        img = records[0]['image']
        display(img)
    # assign tertiles and show distribution
    tert_records = cnn_train.assign_tertiles(records)
    print('Assigned tertiles, sample counts:')
    print(Counter([r['tertile'] for r in tert_records]))
else:
    print('Skipping prepare_records: no data dirs')


In [ ]:
# Dataset / model / training demo

# Build dataset and run a quick smoke training (1 epoch) if samples exist
if len(records) > 0:
    from torchvision.transforms import Compose, ToTensor, Normalize
    transform = Compose([ToTensor(), Normalize([0.5]*3, [0.5]*3)])
    ds = cnn_train.I20Dataset(tert_records, transform=transform)
    print('Dataset samples:', len(ds))
    from torch.utils.data import DataLoader
    dl = DataLoader(ds, batch_size=8, shuffle=True)
    batch = next(iter(dl))
    imgs, labels = batch
    print('Batch shapes:', imgs.shape, labels.shape)
    # quick train call (epochs=1) to smoke test
    model = cnn_train.train(tert_records[:64], epochs=1, batch_size=8)
    print('Finished quick train')
else:
    print('No records to train on')

# Save example
# torch.save(model.state_dict(), 'cnn_rtp20_demo.pth')
